In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval
from ta.volatility import BollingerBands



In [21]:


# Login no TradingView
tv = TvDatafeed()

ticker = 'WDO1!'
exchange = 'BMFBOVESPA'


df = tv.get_hist(
    symbol=ticker,
    exchange=exchange,
    interval=Interval.in_2_hour,
    n_bars=10000
)
df = df[df.index.year >=2000].dropna()
df.index = pd.to_datetime(df.index).normalize().date
df.drop(columns='symbol',inplace=True)
df.index = pd.to_datetime(df.index).normalize()
df.dropna(inplace=True)
df['ret'] = df['close'].pct_change()
df.tail()




,open,high,low,close,volume,ret
2025-12-26,5550.0,5570.0,5543.0,5555.5,637990.0,0.004884
2025-12-26,5555.5,5555.5,5523.0,5538.0,555707.0,-0.003150
2025-12-26,5538.0,5549.0,5528.0,5543.5,321910.0,0.000993
2025-12-26,5543.0,5552.5,5538.0,5547.0,152319.0,0.000631
2025-12-26,5547.0,5550.5,5540.0,5545.5,81673.0,-0.000270


In [22]:
# =========================
# BOLLINGER BANDS
# =========================
bb = BollingerBands(
    close=df["close"],
    window=20,
    window_dev=2
)

df["bb_upper"] = bb.bollinger_hband()
df["bb_lower"] = bb.bollinger_lband()
df["bb_mid"]   = bb.bollinger_mavg()

# =========================
# SINAIS
# =========================
# entradas
df["long_entry"]  = df["close"] < df["bb_lower"]
df["short_entry"] = df["close"] > df["bb_upper"]

# saídas: volta para DENTRO da banda
df["exit_long"] = (
    (df["close"] >= df["bb_lower"]) &
    (df["close"].shift(1) < df["bb_lower"].shift(1))
)

df["exit_short"] = (
    (df["close"] <= df["bb_upper"]) &
    (df["close"].shift(1) > df["bb_upper"].shift(1))
)

# =========================
# POSIÇÃO (regime long / short)
# =========================
position = 0
positions = []

for long_e, short_e, exit_l, exit_s in zip(
    df["long_entry"],
    df["short_entry"],
    df["exit_long"],
    df["exit_short"]
):
    if position == 0:
        if long_e:
            position = 1          # entra long
        elif short_e:
            position = -1         # entra short

    elif position == 1:
        if exit_l:
            position = 0          # sai do long ao voltar p/ dentro da banda

    elif position == -1:
        if exit_s:
            position = 0          # sai do short ao voltar p/ dentro da banda

    positions.append(position)

df["position"] = positions


df["strategy_ret"] = df["position"].shift(1) * df["ret"]


df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()
df.tail()

,open,high,low,close,volume,ret,bb_upper,bb_lower,bb_mid,long_entry,short_entry,exit_long,exit_short,position,strategy_ret,strategy,buy_hold
2025-12-26,5550.0,5570.0,5543.0,5555.5,637990.0,0.004884,5599.136081,5501.263919,5550.20,False,False,False,False,0,0.0,0.154943,0.127235
2025-12-26,5555.5,5555.5,5523.0,5538.0,555707.0,-0.003150,5598.764612,5502.935388,5550.85,False,False,False,False,0,-0.0,0.154943,0.124085
2025-12-26,5538.0,5549.0,5528.0,5543.5,321910.0,0.000993,5598.500934,5504.799066,5551.65,False,False,False,False,0,0.0,0.154943,0.125078
2025-12-26,5543.0,5552.5,5538.0,5547.0,152319.0,0.000631,5598.536607,5505.963393,5552.25,False,False,False,False,0,0.0,0.154943,0.125710
2025-12-26,5547.0,5550.5,5540.0,5545.5,81673.0,-0.000270,5598.487416,5507.112584,5552.80,False,False,False,False,0,-0.0,0.154943,0.125439


In [23]:
df_plot = df.tail(300)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(f"{ticker} Close Price", "Bollinger Bands (20, 2)")
)

# =========================
# CLOSE PRICE + BOLLINGER
# =========================
fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["close"],
        name="Close Price",
        line=dict(color="black")
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["bb_upper"],
        name="Upper Band",
        line=dict(color="red", dash="dash")
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["bb_mid"],
        name="Middle Band",
        line=dict(color="blue")
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["bb_lower"],
        name="Lower Band",
        line=dict(color="green", dash="dash")
    ),
    row=1, col=1
)

# -------------------------
# ENTRADAS
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_plot.index[df_plot["long_entry"]],
        y=df_plot.loc[df_plot["long_entry"], "close"],
        mode="markers",
        name="Long Entry (Below Lower Band)",
        marker=dict(symbol="triangle-up", color="green", size=10)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index[df_plot["short_entry"]],
        y=df_plot.loc[df_plot["short_entry"], "close"],
        mode="markers",
        name="Short Entry (Above Upper Band)",
        marker=dict(symbol="triangle-down", color="red", size=10)
    ),
    row=1, col=1
)

# =========================
# DISTÂNCIA À MÉDIA (diagnóstico)
# =========================
fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["position"],
        name="Price - BB Mid",
        line=dict(color="purple")
    ),
    row=2, col=1
)


# =========================
# LAYOUT
# =========================
fig.update_layout(
    title_text=f"{ticker} | Bollinger Bands Mean Reversion Strategy",
    height=700,
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Distance to BB Mid", row=2, col=1)

fig.show()


In [24]:
# =========================
# PLOT (styled to match previous Plotly figure)
# =========================
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.65, 0.25],
    subplot_titles=(f"{ticker} | Cumulative Returns", "Position")
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2, color="black")
    ),
    row=1, col=1
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2, color="blue")
    ),
    row=1, col=1
)

# Position (filled area)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["position"],
        name="Position",
        mode="lines",
        line=dict(width=1, color="green"),
        fill="tozeroy",
        fillcolor="rgba(0,200,0,0.08)"
    ),
    row=2, col=1
)

# Layout
fig.update_layout(
    title_text=f"{ticker} | MACD",
    template="plotly_white",
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Position", row=2, col=1, range=[-0.05, 1.05])
fig.update_xaxes(showticklabels=False, row=1, col=1)

fig.show()




In [25]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 12.54%
Strategy Return:   15.49%

Buy & Hold Vol: 6.60%
Strategy Vol:   2.21%

Buy & Hold Sharpe: 0.08
Strategy Sharpe:   0.28
